<a href="https://colab.research.google.com/github/mohamadfaisalbashir/grokking-Deep-Learning/blob/main/09_modeling_probabilities_and_nonlinearities.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Chapter 9: Modeling Probabilities and Nonlinearities - Activation Functions**

This notebook summarizes Chapter 9 of *Grokking Deep Learning*:

1. What Is an Activation Function?
2. Standard Hidden-Layer Activation Functions
3. Standard Output Layer Activation Functions
4. The Core Issue: Inputs Have Similarity
5. softmax Computation
6. Activation Installation Instructions
7. Multiplying Delta by the Slope
8. Converting Output to Slope (Derivative)
9. Upgrading the MNIST Network
10. Summary

Earlier chapters already used one activation function, `relu`, inside the first deep neural network. This chapter looks at activation functions as a topic in their own right. It explains what an activation function is and which properties make a function suitable, presents the two standard hidden-layer choices (`sigmoid` and `tanh`) and the three typical kinds of output layer (with `softmax` as the choice for single-label classification), and shows how to "install" any activation function into both forward propagation and backpropagation. At the end, the MNIST network is upgraded with `tanh` in the hidden layer and `softmax` in the output layer.

The chapter opens with a quotation from a letter written by George Gordon Byron to Annabella Milbanke on November 10, 1813: Byron remarks that he knows 2 and 2 make 4 and would happily prove it, but that he would take far greater pleasure in turning 2 and 2 into five.

---
## **1. What Is an Activation Function?**

**An activation function transforms the values of the neurons in a layer during prediction.**

This should already sound familiar, because the `relu` function that was added to the hidden layer of the three-layer network (as in the figure below) is an activation function. Its effect was to turn every negative number into 0.

> A three-layer network (`layer_0`, `layer_1`, `layer_2`) in which the activation function `relu` is applied to the neurons of `layer_1`
>
> <img src="pictures/chapter 9/picture 9.1.png" alt="Figure 9.1" width="600"/>

Put simply, an activation function is any function that takes one number and returns another number. There are, however, infinitely many such functions, and only some of them are useful as activation functions. Four constraints describe what a good activation function looks like; stepping outside them is usually a bad idea, as the following sections show.

### **1.1 Constraint 1: The Function Must Be Continuous and Infinite in Domain**

A proper activation function must produce an output for **any** input: there must be no number that you can feed in and for which no output exists.

> Left: a function that is defined in only four separate spots, so it has no y value for most x values (a poor activation function). Right: `y = x * x`, which is continuous and defined for every input
>
> <img src="pictures/chapter 9/picture 9.2.png" alt="Figure 9.2" width="600"/>

The function on the left is an exaggerated example. It consists of four separate pieces and therefore has no y value for most values of x, which would make it a horrible activation function. The function on the right is continuous and infinite in domain: there is no input x for which an output y cannot be computed.

### **1.2 Constraint 2: Good Activation Functions Are Monotonic, Never Changing Direction**

The second constraint says that the function is 1:1: it must never change direction, that is, it must be either always increasing or always decreasing.

> Left: `y = x * x`, which is not monotonic because it decreases and then increases. Right: `y = x`, which is always increasing
>
> <img src="pictures/chapter 9/picture 9.3.png" alt="Figure 9.3" width="600"/>

The function `y = x * x` on the left is not an ideal activation function, because it is neither always increasing nor always decreasing. A telltale sign is that, for every input except 0, there is a second input (its negative) that produces the same output. The function `y = x` on the right is always increasing, so no two inputs give the same output.

This constraint is **not strictly a requirement**. Unlike functions with missing values (which are not continuous), functions that are not monotonic can still be optimized. The implication of several inputs mapping to the same output is nevertheless worth considering. When a network learns, it searches for weight configurations that give a specific output, and the search becomes harder if there are many right answers, because then the network has many possible perfect configurations.

- An **optimist** would say that it is great: the right answer can be found in many places, so finding it is more likely.
- A **pessimist** would say that it is terrible: there is no longer a single correct direction in which to reduce the error, since moving in either direction could theoretically make progress.

The pessimist's observation turns out to matter more. The chapter suggests that readers who want to study this in depth should look into **convex versus non-convex optimization**, a topic covered by entire university and online courses.

### **1.3 Constraint 3: Good Activation Functions Are Nonlinear (They Squiggle or Turn)**

This constraint recalls the idea of *sometimes correlation* from Chapter 6. To create it, neurons had to be able to correlate **selectively** with their input neurons, so that a very negative signal from one input could reduce how much the neuron correlated with any input (in the case of `relu`, by forcing the neuron to drop to 0).

This phenomenon is made possible by **any function that curves**. Functions that look like straight lines merely **scale** the incoming weighted average. Multiplying by a constant such as 2 does not change how correlated a neuron is with its various inputs; it only makes the overall correlation louder or softer. A straight-line activation does not let one weight influence how correlated the neuron is with the other weights. What is really wanted is *selective* correlation: one incoming signal should be able to raise or lower how correlated the neuron is with all the other incoming signals. All curved lines achieve this, to varying degrees.

> Left: `y = (2 * x) + 5`, a linear function. Right: `y = relu(x)`, a nonlinear function that usually makes a better activation function
>
> <img src="pictures/chapter 9/picture 9.4.png" alt="Figure 9.4" width="600"/>

Accordingly, the function on the left is **linear**, whereas the one on the right (`relu`) is **nonlinear** and will usually make a better activation function. The chapter notes that there are exceptions, which it says will be discussed later.

### **1.4 Constraint 4: Good Activation Functions (and Their Derivatives) Should Be Efficiently Computable**

This constraint is simple: an activation function is called very often (sometimes billions of times), so it must not be slow to compute. Many recent activation functions have become popular precisely because they are cheap to compute, at the expense of expressiveness, and `relu` is a prime example.

---
## **2. Standard Hidden-Layer Activation Functions**

**Of the infinitely many possible functions, which ones are most commonly used?**

Even with the four constraints, an infinite (possibly transfinite) number of functions could serve as activation functions. The last few years have brought progress in state-of-the-art activations, yet a relatively small list of functions still covers the vast majority of needs, and improvements over them have mostly been minute.

### **2.1 `sigmoid` Is the Bread-and-Butter Activation**

`sigmoid` smoothly squishes an infinite range of inputs into an output between 0 and 1. In many circumstances this allows the output of an individual neuron to be interpreted as a **probability**. For this reason the nonlinearity is used in both hidden layers and output layers.

> The `sigmoid` curve: inputs from -6 to 6 are mapped smoothly to values between 0 and 1, with 0.5 at an input of 0
>
> <img src="pictures/chapter 9/picture 9.5.png" alt="Figure 9.5" width="600"/>

### **2.2 `tanh` Is Better than `sigmoid` for Hidden Layers**

Recall the idea of modeling selective correlation. `sigmoid` provides varying degrees of **positive** correlation, which is useful. `tanh` has the same S-shape as `sigmoid`, except that its output lies **between -1 and 1**. It can therefore also contribute some **negative correlation**.

`tanh` is not very useful in output layers (unless the data being predicted lies between -1 and 1), but its ability to express negative correlation is powerful in hidden layers. On many problems, `tanh` outperforms `sigmoid` in hidden layers.

> The `tanh` curve: the same S-shape as `sigmoid`, but with outputs between -1 and 1 and a value of 0 at an input of 0
>
> <img src="pictures/chapter 9/picture 9.6.png" alt="Figure 9.6" width="600"/>


---
## **3. Standard Output Layer Activation Functions**

**Choosing the best one depends on what you are trying to predict.**

What is best for hidden-layer activations can differ considerably from what is best for output-layer activations, especially in classification. Broadly speaking, there are three major types of output layer.

### **3.1 Configuration 1: Predicting Raw Data Values (No Activation Function)**

This is perhaps the most straightforward but the least common type of output layer. Sometimes a network is trained to transform one matrix of numbers into another matrix of numbers whose range (the difference between its lowest and highest values) is something other than a probability. An example is predicting the average temperature in Colorado from the temperatures in the surrounding states.

The main concern is that the output nonlinearity must be able to produce the right answers. A `sigmoid` or `tanh` would be inappropriate here, because it squeezes every prediction into a narrow interval (the book says between 0 and 1), whereas a temperature may take any value. The author states that, for such a task, he would very likely train the network **without an activation function on the output**.

### **3.2 Configuration 2: Predicting Unrelated Yes/No Probabilities (`sigmoid`)**

Often a single network should produce several binary probabilities. This was done in the "Gradient descent with multiple inputs and outputs" section of Chapter 5, where the network predicted whether the team would win, whether there would be injuries, and the morale of the team (happy or sad) from the input data.

As an aside, when a network has hidden layers, predicting several things at once can be beneficial. While learning to predict one label, the network often learns something that is also useful for another label. For example, if the network becomes very good at predicting whether the team wins, the same hidden layer is likely to be very useful for predicting whether the team is happy or sad, and without that extra signal the happiness label might be harder to predict. This varies greatly from problem to problem, but it is worth keeping in mind.

In these situations `sigmoid` is the best choice, because it models an **individual probability separately for each output node**.

### **3.3 Configuration 3: Predicting Which-One Probabilities (`softmax`)**

By far the most common use case is predicting **a single label out of many**. In the MNIST digit classifier, for instance, the goal is to predict *which* number appears in the image, and it is known in advance that an image cannot contain more than one number. One could train such a network with `sigmoid` and declare that the highest output probability is the most likely label, and this works reasonably well. It is far better, however, to use an activation function that builds in the following idea: **as the probability of one label grows, the probability of every other label should shrink**.

Why is this property desirable? Consider how weight updates are performed. Suppose the MNIST classifier should predict that the image is a 9, and suppose the raw weighted sums entering the final layer (before any activation function) are 0.0 for the digits 0 to 8 and 100 for the digit 9. The network's raw input to the last layer predicts 0 for every node except 9, so one might call it perfect. The figure below follows these numbers through `sigmoid` and `softmax` and then shows the mean squared error obtained from the `sigmoid` outputs.

> From top to bottom: the raw dot-product values (0.0 for digits 0 to 8, 100 for digit 9), the `sigmoid` outputs (.50 for digits 0 to 8, .99 for digit 9), the `softmax` outputs (0.0 for digits 0 to 8, 1.0 for digit 9), and the mean squared error of the `sigmoid` outputs (.25 for digits 0 to 8, .00 for digit 9)
>
> <img src="pictures/chapter 9/picture 9.7.png" alt="Figure 9.7" width="600"/>

- **`sigmoid`.** Strangely, the network now appears *less* sure. The digit 9 is still the highest output (.99), but the network seems to believe that there is a 50% chance that the image is any of the other digits (.50 each).
- **`softmax`.** It interprets the same input very differently: the output is 1.0 for the digit 9 and 0.0 for every other digit. The network does not even suspect that the image is one of the other digits.
- **The consequence for backpropagation.** The `sigmoid` behavior may look like a purely theoretical flaw, but it has serious practical effects. The mean squared error on the `sigmoid` outputs is .25 for each wrong digit (the squared difference between .50 and the target of 0) and .00 for the 9. Although the network predicted perfectly, the weights are in line for a **massive update**.

The reason is that, for `sigmoid` to reach zero error, it is not enough to predict the highest positive number for the true label; it must also predict 0 everywhere else. `softmax` asks which digit fits this input best. `sigmoid`, in contrast, insists that the image is the digit 9 and only the digit 9, with nothing in common with any of the other MNIST digits.

---
## **4. The Core Issue: Inputs Have Similarity**

**Different numbers share characteristics, and it is good to let the network believe that.**

MNIST digits are not entirely different from each other: they have overlapping pixel values. The average 2, for example, shares quite a bit with the average 3.

> A 2 and a 3 from MNIST, with the strokes they have in common (the top curve) circled
>
> <img src="pictures/chapter 9/picture 9.8.png" alt="Figure 9.8" width="600"/>

As a general rule, **similar inputs create similar outputs**: if the numbers that are multiplied by a matrix are quite similar, the resulting numbers will be quite similar as well.

Consider the 2 and the 3 in the figure. Suppose the 2 is forward propagated and a small amount of probability accidentally lands on the label 3. What would it mean for the network to treat this as a big mistake and respond with a big weight update? It would **penalize the network for recognizing a 2 by anything other than features that belong exclusively to 2s**. For instance, it would penalize recognizing a 2 from the top curve, because 2 and 3 share the same curve at the top of the image. Training with `sigmoid` penalizes the network for predicting a 2 from this input precisely because it would be looking for the same input that it uses for 3s. As a result, when a 3 arrives, the label 2 gets some probability, since part of the image looks like a 2.

The side effect is that most images share many pixels in the middle of the image, so the network starts to concentrate on the **edges**.

> The weights of a 2-detector node drawn as an image: the middle is muddy, and the heaviest weights are at the end points of the 2 near the edge of the image
>
> <img src="pictures/chapter 9/picture 9.9.png" alt="Figure 9.9" width="600"/>

In the weights of the 2-detector node, the middle of the image looks muddy, and the heaviest weights lie at the end points of the 2, toward the edge of the image. These end points are probably the best *individual* indicators of a 2, but the best detector overall is one that sees the entire shape for what it is. Individual indicators can be triggered accidentally by a 3 that is slightly off-center or tilted the wrong way. The network is not learning the true essence of a 2, because it has to learn "2 and *not* 1, *not* 3, *not* 4", and so on.

What is needed is an output activation that **does not penalize labels that are similar**, and that pays attention to all the information that can be indicative of any potential input. `softmax` has a further advantage: its probabilities always **sum to 1**, so each individual prediction can be read as a global probability that the input is a particular label. In both theory and practice, `softmax` works better.

---
## **5. softmax Computation**

**`softmax` exponentiates every input value and then divides by the sum of the layer.**

The computation is demonstrated on the hypothetical output values from before: raw dot-product values of 0.0 for the digits 0 to 8 and 100 for the digit 9.

### **5.1 Step 1: Raise Each Value Exponentially**

For each value x, compute $e^x$, where $e$ is a special number of about 2.71828. The next figure shows the curve of $e^x$.

> The curve of $e^x$: very small positive values for negative inputs, 1 at an input of 0, and rapidly growing values for positive inputs
>
> <img src="pictures/chapter 9/picture 9.10.png" alt="Figure 9.10" width="600"/>

This step turns every prediction into a **positive number**: negative numbers become very small positive numbers, and big numbers become very big numbers. (Exponential growth, which many people have heard of, refers to this function or one very similar to it.) In the example, all the 0s turn into 1s, because 1 is the y-intercept of $e^x$, and the 100 turns into a massive number, $2.688 \times 10^{43}$ (a 2 followed by 43 zeros). Any negative input would have become a value between 0 and 1.

### **5.2 Step 2: Divide by the Sum of the Layer**

Next, all the nodes of the layer are summed, and each value is divided by that sum. This effectively makes every number 0 except the value for the label 9, which becomes 1.0.

> The softmax computation step by step: the raw values (0.0 for digits 0 to 8, 100 for digit 9), their exponentials (1.0 for digits 0 to 8, about 2.688 * 10^43 for digit 9), and the final softmax output (0.0 for digits 0 to 8, 1.0 for digit 9)
>
> <img src="pictures/chapter 9/picture 9.11.png" alt="Figure 9.11" width="600"/>

### **5.3 Properties of softmax**

- The higher the network predicts one value, the lower it predicts all the others. This increases what the book calls the **sharpness of attenuation** and encourages the network to predict one output with very high probability.
- To adjust how aggressively this happens, one can exponentiate with a number slightly higher or lower than $e$. Lower numbers give lower attenuation, and higher numbers give higher attenuation. Most people simply keep $e$.

---
## **6. Activation Installation Instructions**

**How do you add your favorite activation function to any layer?**

Having covered a wide variety of activation functions and their usefulness in hidden and output layers, the chapter turns to the proper way of installing one in a network. Fortunately, an example has already been seen in the first deep neural network: a `relu` activation function was added to the hidden layer. In forward propagation this was straightforward. The values that `layer_1` would have had without an activation were passed through the `relu` function:

*Note: the three lines below, and the listing further down, are excerpts of the earlier `relu` network (they use `images`, `i`, `weights_0_1`, `weights_1_2`, `alpha`, `labels`, and so on, which are defined in that earlier network and not in this chapter). They are therefore shown as plain text and not as runnable cells. The book highlights the call to `relu` in the second line in bold.*

```python
layer_0 = images[i:i+1]
layer_1 = relu(np.dot(layer_0,weights_0_1))
layer_2 = np.dot(layer_1,weights_1_2)
```

A bit of terminology: the **input to a layer** is the value *before* the nonlinearity. Here, the input to `layer_1` is `np.dot(layer_0,weights_0_1)`. It should not be confused with the previous layer, `layer_0`.

### **6.1 Compensating for the Activation in Backpropagation**

Adding an activation function to forward propagation is easy, but properly compensating for it in backpropagation is more subtle.

In Chapter 6, an interesting operation was used to create the `layer_1_delta` variable: wherever `relu` had forced a `layer_1` value to 0, the delta was also multiplied by 0. The reasoning was that a `layer_1` value of 0 had no effect on the output prediction, so it should have no impact on the weight update either, since it was not responsible for the error. This is the **extreme form of a more nuanced property**, which becomes visible when one looks at the shape of `relu`.

> The `relu` function: a slope of 0 for negative inputs and a slope of 1 for positive inputs
>
> <img src="pictures/chapter 9/picture 9.12.png" alt="Figure 9.12" width="600"/>

The slope of `relu` is exactly 1 for positive numbers and exactly 0 for negative numbers. A tiny change to the input of the function has a 1:1 effect if the neuron was predicting positively and a 0:1 effect (none at all) if it was predicting negatively. In other words, **the slope measures how much the output of `relu` changes when its input changes**.

The purpose of a delta at this point is to tell earlier layers to make this node's input higher or lower next time. The slope is useful here because it adjusts the delta that is backpropagated from the following layer so that it takes into account whether this node actually contributed to the error.

Hence, to generate `layer_1_delta` during backpropagation, the delta that is backpropagated from `layer_2` (`layer_2_delta.dot(weights_1_2.T)`) is **multiplied by the slope of `relu` at the point predicted in forward propagation**. For some deltas that slope is 1 (positive values), and for others it is 0 (negative values). The book prints the following listing, with the multiplication by `relu2deriv(layer_1)` highlighted in bold, together with two annotations on the functions at the bottom:

```python
error += np.sum((labels[i:i+1] - layer_2) ** 2)

correct_cnt += int(np.argmax(layer_2) == \
                                np.argmax(labels[i:i+1]))

layer_2_delta = (labels[i:i+1] - layer_2)
layer_1_delta = layer_2_delta.dot(weights_1_2.T)\
                            * relu2deriv(layer_1)

weights_1_2 += alpha * layer_1.T.dot(layer_2_delta)
weights_0_1 += alpha * layer_0.T.dot(layer_1_delta)

def relu(x):
    return (x >= 0) * x

def relu2deriv(output):
    return output >= 0
```

The book's annotations say that `relu` returns `x` if `x > 0` and returns 0 otherwise, and that `relu2deriv` returns 1 for an input greater than 0 and 0 otherwise.

`relu2deriv` is a special function that takes the **output** of `relu` and computes the slope of `relu` at that point (for all values of the output vector at once). This raises a question: how can similar adjustments be made for all the other nonlinearities that are not `relu`? Consider `relu` and `sigmoid` side by side.

> The `relu` function (left) and the `sigmoid` function (right), shown side by side to compare how their slopes behave
>
> <img src="pictures/chapter 9/picture 9.13.png" alt="Figure 9.13" width="600"/>

The important point in these figures is that the **slope indicates how much a tiny change to the input affects the output**. The delta arriving from the following layer has to be modified so that it takes into account whether a weight update before this node would have any effect at all. The goal is to adjust weights in order to reduce the error, and this step encourages the network to **leave weights alone when adjusting them would have little or no effect**, which it does by multiplying by the slope. The procedure is no different for `sigmoid`.

---
## **7. Multiplying Delta by the Slope**

**To compute `layer_delta`, multiply the backpropagated delta by the layer's slope.**

`layer_1_delta[0]` represents how much higher or lower the first hidden node of `layer_1` should be in order to reduce the network's error for a particular training example. When there is no nonlinearity, it is simply the weighted average of the deltas of `layer_2`.

> A network with an input layer, a hidden layer, and a prediction node: `layer_1_delta[0]` belongs to the first hidden node, and the weights coming into that node are the ones being informed by it
>
> <img src="pictures/chapter 9/picture 9.14.png" alt="Figure 9.14" width="600"/>

The end goal of the delta on a neuron is to inform the weights **whether they should move**. If moving them would have no effect, they (as a group) should be left alone. This is obvious for `relu`, which is either fully on or fully off (the book repeats the `relu` and `sigmoid` graphs of the previous figure at this point). `sigmoid` is somewhat more nuanced.

### **7.1 What the `sigmoid` Slope Means**

Consider a single `sigmoid` neuron. Its sensitivity to changes in the input increases gradually as the input approaches 0 from either direction, whereas very positive and very negative inputs have a slope very close to 0. Consequently, as the input becomes very positive or very negative, small changes to the incoming weights matter less and less for the neuron's error on this training example.

In broader terms, many hidden nodes are irrelevant to the accurate prediction of a particular input, for instance a 2 (perhaps they are used only for 8s). Their weights should not be disturbed too much, because that could corrupt their usefulness elsewhere.

### **7.2 Stickiness**

The same property also creates a notion of **stickiness**. Weights that have been updated a lot in one direction (for similar training examples) confidently predict a high or a low value. Such nonlinearities make it harder for an occasional erroneous training example to corrupt knowledge that has been reinforced many times.

---
## **8. Converting Output to Slope (Derivative)**

**Most great activations can convert their output to their slope (an efficiency win).**

Now that it is clear that adding an activation to a layer changes how that layer's delta is computed, the question is how this is done efficiently in practice. The new operation needed is the **derivative** of whichever nonlinearity was used.

Readers who know calculus may be surprised by how the popular nonlinearities do this. Instead of computing the derivative at a given point of the curve in the usual way, most great activation functions provide a way to compute the derivative **from the output of the layer produced during forward propagation**. This has become standard practice for computing derivatives in neural networks, and it is quite handy.

The table below pairs the functions seen so far with their derivatives. Here `input` is a NumPy vector (the input to a layer), `output` is the prediction of the layer, `deriv` is the vector of activation derivatives (the derivative of the activation at each node), and `true` is the vector of true values (typically 1 for the correct label position and 0 everywhere else). The table is reproduced as the book prints it.

| Function | Forward prop | Backprop delta |
|---|---|---|
| `relu` | `ones_and_zeros = (input > 0)`<br>`output = input*ones_and_zeros` | `mask = output > 0`<br>`deriv = output * mask` |
| `sigmoid` | `output = 1/(1 + np.exp(-input))` | `deriv = output*(1-output)` |
| `tanh` | `output = np.tanh(input)` | `deriv = 1 - (output**2)` |
| `softmax` | `temp = np.exp(input)`<br>`output /= np.sum(temp)` | `temp = (output - true)`<br>`output = temp/len(true)` |

The delta computation for `softmax` is special, because `softmax` is used only for the last layer. There is more going on theoretically than the chapter has time to discuss. For now, the next step is to install some better activation functions in the MNIST classification network.

---
## **9. Upgrading the MNIST Network**

**Let's upgrade the MNIST network to reflect what has been learned.**

In theory, `tanh` should make a better hidden-layer activation and `softmax` a better output-layer activation. When they are tested, they do reach a higher score, but, as the author points out, things are not always as simple as they seem. He had to make a few adjustments to tune the network properly for the new activations:

1. **A narrower weight initialization for `tanh`.** Weights are initialized randomly. `np.random.random` creates a random matrix with values spread between 0 and 1; multiplying by 0.2 and subtracting 0.1 rescales this range to between -0.1 and 0.1. That worked well for `relu` but is less optimal for `tanh`, which prefers a narrower random initialization, so the input weights were changed to lie between -0.01 and 0.01.
2. **The error calculation was removed.** `softmax` is technically best used with an error function called **cross entropy**. The network computes `layer_2_delta` correctly for this error measure, but because the chapter has not yet analyzed why this error function is advantageous, the lines that compute it were left out.
3. **`alpha` had to be retuned.** As with almost any change to a neural network, `alpha` needed to be revisited. A much higher `alpha` was required to reach a good score within 300 iterations.

With these changes, the network reached a higher test accuracy of 87%, as expected.

*Data note:* the first cell obtains MNIST through Keras (`from keras.datasets import mnist`), so the `keras` package must be installed, and the dataset is downloaded the first time `mnist.load_data()` is called. The network is trained on the first 1,000 training images and evaluated on the full test set. The first cell loads the data, converts the labels into one-hot matrices (`labels` and `test_labels`), and defines `tanh`, `tanh2deriv`, and `softmax`. The function `tanh2deriv` takes the *output* of `tanh` and returns `1 - (output ** 2)`, in line with the table of the previous section. In `softmax`, the sum is taken along `axis=1`, so every row (one row per image in a batch) is normalized separately and sums to 1.

In [1]:
import numpy as np, sys
np.random.seed(1)

from keras.datasets import mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()

images, labels = (x_train[0:1000].reshape(1000,28*28)\
                                / 255, y_train[0:1000])
one_hot_labels = np.zeros((len(labels),10))
for i,l in enumerate(labels):
    one_hot_labels[i][l] = 1
labels = one_hot_labels

test_images = x_test.reshape(len(x_test),28*28) / 255
test_labels = np.zeros((len(y_test),10))
for i,l in enumerate(y_test):
    test_labels[i][l] = 1

def tanh(x):
    return np.tanh(x)
def tanh2deriv(output):
    return 1 - (output ** 2)
def softmax(x):
    temp = np.exp(x)
    return temp / np.sum(temp, axis=1, keepdims=True)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


The second cell sets the hyperparameters (`alpha = 2`, 300 iterations, 100 hidden nodes, batches of 100 images), initializes the weights (`weights_0_1` between -0.01 and 0.01, `weights_1_2` between -0.1 and 0.1), and trains the network. The forward pass uses `tanh` in the hidden layer and `softmax` in the output layer, and `layer_1_delta` is the backpropagated delta multiplied by `tanh2deriv(layer_1)`. After each iteration the network is evaluated on the whole test set, and every 10 iterations the test accuracy and the training accuracy are printed. The listing also contains lines that use a `dropout_mask`; the chapter does not discuss them, so they are reproduced unchanged and not elaborated here. Running the cell takes a while, because the test set is evaluated one image at a time in each of the 300 iterations.

In [2]:
alpha, iterations, hidden_size = (2, 300, 100)
pixels_per_image, num_labels = (784, 10)
batch_size = 100

weights_0_1 = 0.02*np.random.random((pixels_per_image,hidden_size))-0.01
weights_1_2 = 0.2*np.random.random((hidden_size,num_labels)) - 0.1

for j in range(iterations):
    correct_cnt = 0
    for i in range(int(len(images) / batch_size)):
        batch_start, batch_end=((i * batch_size),((i+1)*batch_size))
        layer_0 = images[batch_start:batch_end]
        layer_1 = tanh(np.dot(layer_0,weights_0_1))
        dropout_mask = np.random.randint(2,size=layer_1.shape)
        layer_1 *= dropout_mask * 2
        layer_2 = softmax(np.dot(layer_1,weights_1_2))

        for k in range(batch_size):
            correct_cnt += int(np.argmax(layer_2[k:k+1]) == \
                               np.argmax(labels[batch_start+k:batch_start+k+1]))
        layer_2_delta = (labels[batch_start:batch_end]-layer_2)\
                        / (batch_size * layer_2.shape[0])
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) \
                        * tanh2deriv(layer_1)
        layer_1_delta *= dropout_mask

        weights_1_2 += alpha * layer_1.T.dot(layer_2_delta)
        weights_0_1 += alpha * layer_0.T.dot(layer_1_delta)
    test_correct_cnt = 0

    for i in range(len(test_images)):

        layer_0 = test_images[i:i+1]
        layer_1 = tanh(np.dot(layer_0,weights_0_1))
        layer_2 = np.dot(layer_1,weights_1_2)
        test_correct_cnt += int(np.argmax(layer_2) == \
                                np.argmax(test_labels[i:i+1]))
    if(j % 10 == 0):
        sys.stdout.write("\n"+ "I:" + str(j) + \
         " Test-Acc:"+str(test_correct_cnt/float(len(test_images)))+\
         " Train-Acc:" + str(correct_cnt/float(len(images))))


I:0 Test-Acc:0.4113 Train-Acc:0.168
I:10 Test-Acc:0.7043 Train-Acc:0.724
I:20 Test-Acc:0.7176 Train-Acc:0.732
I:30 Test-Acc:0.7414 Train-Acc:0.763
I:40 Test-Acc:0.7712 Train-Acc:0.797
I:50 Test-Acc:0.7972 Train-Acc:0.821
I:60 Test-Acc:0.8123 Train-Acc:0.844
I:70 Test-Acc:0.823 Train-Acc:0.851
I:80 Test-Acc:0.8301 Train-Acc:0.873
I:90 Test-Acc:0.8346 Train-Acc:0.895
I:100 Test-Acc:0.8391 Train-Acc:0.883
I:110 Test-Acc:0.8432 Train-Acc:0.896
I:120 Test-Acc:0.8453 Train-Acc:0.903
I:130 Test-Acc:0.8486 Train-Acc:0.905
I:140 Test-Acc:0.8509 Train-Acc:0.909
I:150 Test-Acc:0.8534 Train-Acc:0.915
I:160 Test-Acc:0.8559 Train-Acc:0.919
I:170 Test-Acc:0.8572 Train-Acc:0.925
I:180 Test-Acc:0.859 Train-Acc:0.924
I:190 Test-Acc:0.8611 Train-Acc:0.921
I:200 Test-Acc:0.8623 Train-Acc:0.927
I:210 Test-Acc:0.863 Train-Acc:0.931
I:220 Test-Acc:0.8635 Train-Acc:0.932
I:230 Test-Acc:0.8646 Train-Acc:0.94
I:240 Test-Acc:0.8654 Train-Acc:0.945
I:250 Test-Acc:0.8662 Train-Acc:0.929
I:260 Test-Acc:0.8669 Trai

The test accuracy rises from 0.4113 at iteration 0 to 0.8689 at iteration 290, while the training accuracy reaches about 0.943. A fresh run reproduces these numbers, except that the book prints a few training accuracies in a shortened form.

---
## **10. Summary**

- An **activation function** is a function applied to the neurons of a layer during prediction. Of the infinitely many possible functions, good activation functions satisfy four constraints: they are **continuous and infinite in domain**, **monotonic** (never changing direction; this is not strictly required, but multiple inputs mapping to one output make learning harder), **nonlinear** (they curve, which allows selective correlation, whereas straight lines only scale the weighted average), and **efficiently computable**, together with their derivatives.
- For hidden layers, `sigmoid` is the bread-and-butter choice: it squishes any input into the range 0 to 1, so outputs can be read as probabilities. `tanh` has the same shape but ranges from -1 to 1, so it can also express **negative correlation**, and it often outperforms `sigmoid` in hidden layers.
- There are three kinds of output layer. To predict **raw data values** (such as a temperature), use **no activation function**. To predict several **unrelated yes/no probabilities**, use `sigmoid`, which models each output node separately. To predict **which one** of many labels applies (the most common case), use `softmax`.
- `softmax` is preferred for single-label classification because it builds in the idea that a higher probability for one label means lower probabilities for all the others. With `sigmoid`, a perfectly confident prediction (raw values of 0 and 100) still produces large errors on the other labels (.25 each), triggering big and unwarranted weight updates. Because inputs such as MNIST digits **share features**, penalizing similar labels pushes the network to focus on edge pixels instead of the true shape of a digit. `softmax` avoids this and also makes the outputs **sum to 1**.
- `softmax` raises each value to the power $e$ (turning everything positive and magnifying big values; the 100 in the example becomes about $2.688 \times 10^{43}$) and then divides by the sum of the layer. Using a base slightly lower or higher than $e$ changes the sharpness of attenuation, but most people keep $e$.
- To **install** an activation in a layer, apply it to the layer's input (the value before the nonlinearity) in forward propagation, and in backpropagation **multiply the delta arriving from the next layer by the slope of the activation** at the predicted point. For `relu` the slope is 1 or 0; for `sigmoid` it approaches 0 for very positive or very negative inputs, which keeps irrelevant weights from changing and makes well-reinforced weights "sticky".
- Most popular activations can compute their **derivative from their output**, which is efficient: `sigmoid` uses `output*(1-output)`, `tanh` uses `1 - (output**2)`, and `relu` uses a 0/1 mask. The `softmax` delta (`output - true`, scaled by the size of `true`) is special because `softmax` is used only in the last layer, and it is the proper delta for the cross-entropy error.
- In the upgraded MNIST network, `tanh` in the hidden layer and `softmax` in the output layer required a narrower weight initialization (-0.01 to 0.01 for the first weight matrix), removal of the error calculation (cross entropy has not been analyzed yet), and a much higher `alpha` (2). The network reached a test accuracy of about **87%** after 300 iterations.